# Analiza metryk z eksperymentów

Notebook służy do tworzenia wykresów z plików zapisanych w folderach `metrics/`, np.:

```text
outputs/
└── nazwa_eksperymentu/
    ├── config.json
    └── metrics/
        ├── history_by_epoch.csv
        ├── best_validation_metrics.csv
        ├── test_summary.csv
        ├── test_classification_report.csv
        └── test_confusion_matrix.csv
```

Można załadować jeden lub wiele folderów eksperymentów. Dla jednej metryki wykres może zawierać kilka modeli jednocześnie.


In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

PROJ_DIR = Path("/home/jakjas3894/thesis")

if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))
    
from src.archive import extract_training_archives
from src.metrics_plotting import (
    load_histories,
    load_test_summaries,
    load_best_validation,
    available_history_metrics,
    plot_metric,
    plot_metric_grid,
    plot_train_val_pair,
    plot_test_metric_bar,
    plot_confusion_matrix,
)

In [ ]:
PD_PATH = Path(os.environ.get("PDDIRS", "/lustre/pd03/hpc-jakjas3894-1777231388"))
ARCHIVES_DIR = PD_PATH / "models" / "seeds"
TMP_DIR = Path(os.environ.get("TMPDIR", "/tmp"))

print("PD_PATH", PD_PATH)
print("ARCHIVES_DIR", ARCHIVES_DIR)
print("TMP_DIR", TMP_DIR)


# tutaj wstawiamy nazwy archiwów z modelami, których metryki chcemy umieścić na wykresach
experiment_dirs = extract_training_archives(
    archives_dir=ARCHIVES_DIR,
    archive_names=[
        "convnext_tiny_ce_50_288_seed_42.tar.gz",
        "resnet50_baseline_ce_50_288_seed_42.tar.gz",
        "swin_transformer_v2_tiny_ce_50_288_seed_42.tar.gz",
        "modernstem_eca_resnet50_ce_50_288_seed_42.tar.gz",
        "modernstem_eca_msf_resnet50_ce_50_288_seed_42.tar.gz"
        
    ],
    extract_dir=TMP_DIR
)

experiment_dirs

In [ ]:
# i-ta nazwa zostanie przypisana do i-tego archiwum modelu
EXPERIMENT_NAMES = [
    "ConvNeXt Tiny",
    "ResNet-50",
    "Swin Transformer V2 Tiny",
    "RybNet V1",
    "RybNet V2"
]

PLOTS_DIR = Path("plots_metric_comparison")

PLOTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Wczytanie historii treningu

Ładowany jest plik:

```text
metrics/history_by_epoch.csv
```


In [ ]:
history_df = load_histories(
    experiment_dirs=experiment_dirs,
    experiment_names=EXPERIMENT_NAMES if EXPERIMENT_NAMES else None,
)

print("Wczytane eksperymenty:")
print(history_df["experiment"].unique())

print("\nDostępne metryki:")
available_history_metrics(history_df)

## 3. Podgląd danych

In [ ]:
history_df.head()

## 4. Wykres jednej metryki dla wielu modeli

Na jednym wykresie zostaną pokazane przebiegi tej samej metryki dla wszystkich wczytanych eksperymentów.


In [ ]:
plot_metric(
    history_df=history_df,
    metric="val_macro_f1",
    output_dir=PLOTS_DIR,
)

## 5. Porównanie train vs val


In [ ]:
plot_train_val_pair(
    history_df=history_df,
    train_metric="train_macro_f1",
    val_metric="val_macro_f1",
    output_dir=PLOTS_DIR,
)

## 6. Wykresy dla wielu metryk

Wygodne do automatycznego wygenerowania najważniejszych wykresów do analizy.


In [ ]:
METRICS_TO_PLOT = [
    "train_loss",
    "val_loss",
    "train_acc",
    "val_acc",
    "train_f1",
    "val_f1",
    "train_macro_f1",
    "val_macro_f1",
    "train_recall",
    "val_recall",
    "train_macro_recall",
    "val_macro_recall",
    "train_balanced_acc",
    "val_balanced_acc",
]

saved_paths = plot_metric_grid(
    history_df=history_df,
    metrics=METRICS_TO_PLOT,
    output_dir=PLOTS_DIR,
)

# print(history_df.to_csv(sep="\t", index=False))

## 7. Najlepsze wyniki walidacyjne

Ładowany jest plik:

```text
metrics/best_validation_metrics.csv
```


In [ ]:
best_val_df = load_best_validation(
    experiment_dirs=experiment_dirs,
    experiment_names=EXPERIMENT_NAMES if EXPERIMENT_NAMES else None,
)

# print(best_val_df.to_csv(sep="\t", index=False))
best_val_df

## 8. Wyniki testowe

Ładowany jest plik:

```text
metrics/test_summary.csv
```


In [ ]:
test_df = load_test_summaries(
    experiment_dirs=experiment_dirs,
    experiment_names=EXPERIMENT_NAMES if EXPERIMENT_NAMES else None,
)

# print(test_df.to_csv(sep="\t", index=False))
test_df

## 9. Porównanie modeli na zbiorze testowym

In [ ]:
if not test_df.empty:
    plot_test_metric_bar(
        test_df=test_df,
        metric="test_macro_f1",
        output_dir=PLOTS_DIR,
    )

In [ ]:
if not test_df.empty:
    for metric in [
        "test_acc",
        "test_f1",
        "test_macro_f1",
        "test_recall",
        "test_macro_recall",
        "test_balanced_acc",
    ]:
        if metric in test_df.columns:
            plot_test_metric_bar(
                test_df=test_df,
                metric=metric,
                output_dir=PLOTS_DIR,
            )

## 10. Macierz pomyłek dla jednego eksperymentu

Funkcja wczytuje:

```text
metrics/test_confusion_matrix.csv
```


In [ ]:
if experiment_dirs:
    selected_experiment = Path(experiment_dirs[0])
    cm_path = selected_experiment / "metrics" / "test_confusion_matrix.csv"

    if cm_path.exists():
        plot_confusion_matrix(
            confusion_matrix_csv=cm_path,
            output_path=PLOTS_DIR / f"{selected_experiment.name}_confusion_matrix.png",
            normalize=False,
            show_values=False,
        )

        plot_confusion_matrix(
            confusion_matrix_csv=cm_path,
            output_path=PLOTS_DIR / f"{selected_experiment.name}_confusion_matrix_normalized.png",
            normalize=True,
            show_values=False,
        )
    else:
        print(f"Brak pliku: {cm_path}")

## 11. Eksport zbiorczych tabel

In [ ]:
history_df.to_csv(PLOTS_DIR / "combined_history_by_epoch.csv", index=False)

if not best_val_df.empty:
   best_val_df.to_csv(PLOTS_DIR / "combined_best_validation_metrics.csv", index=False)

if not test_df.empty:
   test_df.to_csv(PLOTS_DIR / "combined_test_summary.csv", index=False)

PLOTS_DIR